# Reddit Sarcasm Detection: Baselines, Context, and Error Analysis

Author: Lucy Giang (Aug. 2026)

This project is an NLP study of the [SARC-based Reddit dataset](https://www.kaggle.com/datasets/danofer/sarcasm). We compare a fast TF-IDF + Logistic Regression baseline with fine-tuned BERT, test the value of parent-comment context, and inspect errors across text and Reddit metadata.

**Experiments:** stratified 80/10/10 train/validation/test split; baseline and BERT share the same split; BERT is evaluated with and without parent context. Test metrics are reported only after model selection on validation F1.

> Runtime depends on sample size and hardware. The context ablation adds a second BERT training run. Set `N_SAMPLES` to control cost.


## 1. Setup & Configuration

Install the dependencies (uncomment if running in a fresh environment), import libraries, and define all hyperparameters in one place so experiments are easy to tweak.

In [ ]:
# !pip install -q kagglehub transformers scikit-learn pandas matplotlib seaborn torch

In [ ]:
import os
import random
import time
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    BertForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

sns.set_theme(style="whitegrid")

In [ ]:
# ---------------- Configuration ----------------
SEED            = 42
MODEL_NAME      = "bert-base-uncased"

N_SAMPLES       = 100_000   # subsample size (set to None to use the full dataset)
USE_PARENT      = False     # True -> feed the parent comment as context (sentence-pair input)
MAX_LEN         = 64        # Reddit comments are short; 64 tokens covers most of them
                            # (raise to ~128 if USE_PARENT=True)

BATCH_SIZE      = 32
EPOCHS          = 3
LEARNING_RATE   = 2e-5
WEIGHT_DECAY    = 0.01
WARMUP_RATIO    = 0.1

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"   # mixed precision speeds up training on GPU

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print(f"Device: {DEVICE} | AMP: {USE_AMP}")

## 2. Download and Load the Dataset

We download the dataset with `kagglehub`. The main file we need is **`train-balanced-sarcasm.csv`**, a class-balanced set of Reddit comments with these columns:

| Column | Description |
|---|---|
| `label` | 1 = sarcastic, 0 = not sarcastic |
| `comment` | The comment text (our model input) |
| `author`, `subreddit` | Metadata |
| `score`, `ups`, `downs` | Voting information |
| `date`, `created_utc` | Timestamps |
| `parent_comment` | The comment being replied to (useful context) |

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("danofer/sarcasm")
print("Path to dataset files:", path)
print(os.listdir(path))

In [ ]:
csv_path = os.path.join(path, "train-balanced-sarcasm.csv")
df = pd.read_csv(csv_path)
print("Shape:", df.shape)
df.head()

## 3. Exploratory Data Analysis

Before modelling, we check data quality (missing values, duplicates), class balance, text lengths, and how sarcasm varies by subreddit.

In [ ]:
print(df.info())
print("\nMissing values per column:")
print(df.isna().sum())
print("\nDuplicate comments:", df["comment"].duplicated().sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class balance
sns.countplot(x="label", data=df, ax=axes[0])
axes[0].set_title("Class distribution (0 = not sarcastic, 1 = sarcastic)")

# Comment length (in words)
df["n_words"] = df["comment"].fillna("").str.split().str.len()
sns.histplot(data=df[df["n_words"] <= 100], x="n_words", hue="label", bins=50,
             ax=axes[1], element="step")
axes[1].set_title("Comment length (words, clipped at 100)")

plt.tight_layout()
plt.show()

print(df.groupby("label")["n_words"].describe())

In [ ]:
# Subreddits with the most data and their sarcasm rate
top = (df.groupby("subreddit")["label"]
         .agg(["count", "mean"])
         .query("count >= 2000")
         .sort_values("mean", ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
top.head(10)["mean"].plot.barh(ax=axes[0], color="tab:red")
axes[0].set_title("Highest sarcasm rate (subreddits, n >= 2000)")
axes[0].invert_yaxis()
top.tail(10)["mean"].plot.barh(ax=axes[1], color="tab:blue")
axes[1].set_title("Lowest sarcasm rate (subreddits, n >= 2000)")
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# A few random examples from each class
pd.set_option("display.max_colwidth", 200)
for lbl, name in [(1, "SARCASTIC"), (0, "NOT SARCASTIC")]:
    print(f"--- {name} ---")
    for t in df[df["label"] == lbl]["comment"].dropna().sample(5, random_state=SEED):
        print("  *", t)

## 4. Cleaning, Sampling and Splitting

Steps:

1. Drop rows with a missing comment and remove duplicate comments (duplicates could leak between train and test).
2. Optionally **subsample** to `N_SAMPLES` rows (stratified by label) to keep training time manageable.
3. Split into **80 % train / 10 % validation / 10 % test**, stratified by label so every split stays balanced.

We deliberately do **not** lowercase or strip punctuation by hand: BERT's tokenizer handles casing (for the uncased model) and punctuation/emphasis such as `!!!`, `?`, or `/s` can be genuine sarcasm cues.

In [ ]:
metadata_cols = [c for c in ["subreddit", "score", "ups", "downs"] if c in df.columns]
data = df[["label", "comment", "parent_comment"] + metadata_cols].copy()
data = data.dropna(subset=["comment"])
data = data.drop_duplicates(subset=["comment"]).reset_index(drop=True)
data["parent_comment"] = data["parent_comment"].fillna("")
print("After cleaning:", data.shape)

if N_SAMPLES is not None and N_SAMPLES < len(data):
    data, _ = train_test_split(
        data, train_size=N_SAMPLES, stratify=data["label"], random_state=SEED
    )
    data = data.reset_index(drop=True)
print("After sampling:", data.shape)

train_df, temp_df = train_test_split(
    data, test_size=0.2, stratify=data["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label"], random_state=SEED
)
train_df, val_df, test_df = [d.reset_index(drop=True) for d in (train_df, val_df, test_df)]

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:>5}: {len(d):>7} rows | sarcastic ratio = {d['label'].mean():.3f}")

## 5. Fast Baseline: TF-IDF + Logistic Regression

Fit the baseline on the exact training split used for BERT. Latency is average per-comment prediction time on the held-out test set (model inference only; excludes vectorizer/model loading).

In [ ]:
baseline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=300_000,
                              sublinear_tf=True, strip_accents="unicode")),
    ("clf", LogisticRegression(max_iter=1000, solver="liblinear", random_state=SEED)),
])
baseline.fit(train_df["comment"].fillna(""), train_df["label"])
t0 = time.perf_counter()
baseline_preds = baseline.predict(test_df["comment"].fillna(""))
baseline_latency_ms = (time.perf_counter() - t0) * 1000 / len(test_df)
_, _, baseline_f1, _ = precision_recall_fscore_support(test_df["label"], baseline_preds,
                                                        average="binary", zero_division=0)
baseline_metrics = {"accuracy": accuracy_score(test_df["label"], baseline_preds),
                    "f1": baseline_f1, "inference_ms_per_comment": baseline_latency_ms}
pd.DataFrame([{"model": "TF-IDF + Logistic Regression", **baseline_metrics}])

## 6. Tokenization

BERT does not read raw text. The **WordPiece tokenizer** splits text into sub-word tokens, adds the special `[CLS]` and `[SEP]` tokens, and maps everything to integer IDs. It also produces an **attention mask** that tells the model which positions are real tokens vs. padding.

- Sequences are truncated to `MAX_LEN`.
- Padding is done **dynamically per batch** (via `DataCollatorWithPadding`) so short batches are not padded to the maximum length. This saves a lot of compute.
- If `USE_PARENT=True`, the parent comment is passed as the first segment and the reply as the second segment (sentence-pair input), since sarcasm is often only recognisable in context.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Quick look at what the tokenizer does
sample = "Oh great, another Monday. Just what I needed."
enc = tokenizer(sample)
print(tokenizer.convert_ids_to_tokens(enc["input_ids"]))
print(enc["input_ids"])

In [ ]:
def encode_df(frame):
    if USE_PARENT:
        return tokenizer(
            frame["parent_comment"].tolist(),
            frame["comment"].tolist(),
            truncation=True,
            max_length=MAX_LEN,
        )
    return tokenizer(
        frame["comment"].tolist(),
        truncation=True,
        max_length=MAX_LEN,
    )

class SarcasmDataset(Dataset):
    def __init__(self, frame):
        self.encodings = encode_df(frame)
        self.labels = frame["label"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

t0 = time.time()
train_ds = SarcasmDataset(train_df)
val_ds   = SarcasmDataset(val_df)
test_ds  = SarcasmDataset(test_df)
print(f"Tokenized in {time.time() - t0:.1f}s")

collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collator)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collator)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collator)

lens = [len(x) for x in train_ds.encodings["input_ids"]]
print(f"Token length -> mean: {np.mean(lens):.1f}, 95th pct: {np.percentile(lens, 95):.0f}, "
      f"truncated at {MAX_LEN}: {np.mean(np.array(lens) >= MAX_LEN):.1%}")

## 7. Model

We use `BertForSequenceClassification`: the pre-trained BERT encoder plus a small linear classification layer on top of the pooled `[CLS]` representation, with 2 output classes. During fine-tuning **all** weights are updated, starting from the pre-trained values, which is what lets BERT adapt its general language understanding to the sarcasm task.

In [ ]:
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params / 1e6:.1f}M")

## 8. Training

Training setup:

- **Optimizer:** AdamW with weight decay (not applied to biases or LayerNorm weights).
- **Learning-rate schedule:** linear warmup for the first 10 % of steps, then linear decay to 0. This is the standard recipe for fine-tuning BERT and makes training stable.
- **Loss:** cross-entropy (computed internally by the model when `labels` are provided). The classes are balanced, so no class weighting is needed.
- **Mixed precision (AMP)** on GPU for speed and lower memory.
- **Gradient clipping** (max norm 1.0) to avoid exploding gradients.
- After each epoch we evaluate on the validation set and keep the **best checkpoint** (by validation F1).

In [ ]:
no_decay = ["bias", "LayerNorm.weight"]
param_groups = [
    {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
     "weight_decay": WEIGHT_DECAY},
    {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
     "weight_decay": 0.0},
]
optimizer = AdamW(param_groups, lr=LEARNING_RATE)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_RATIO * total_steps),
    num_training_steps=total_steps,
)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
print(f"Steps per epoch: {len(train_loader)} | total steps: {total_steps}")

In [ ]:
def move(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}

@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    for batch in loader:
        batch = move(batch)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            out = model(**batch)
        total_loss += out.loss.item() * batch["labels"].size(0)
        all_preds.append(out.logits.argmax(dim=-1).cpu())
        all_labels.append(batch["labels"].cpu())
    preds  = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    return {
        "loss": total_loss / len(labels),
        "acc": accuracy_score(labels, preds),
        "precision": p, "recall": r, "f1": f1,
    }, preds, labels


def train_one_epoch(epoch):
    model.train()
    running, n_seen = 0.0, 0
    for step, batch in enumerate(train_loader, 1):
        batch = move(batch)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            out = model(**batch)
            loss = out.loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        bs = batch["labels"].size(0)
        running += loss.item() * bs
        n_seen += bs
        if step % 200 == 0 or step == len(train_loader):
            print(f"  epoch {epoch} step {step}/{len(train_loader)} - train loss {running / n_seen:.4f}")
    return running / n_seen

In [ ]:
history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": []}
best_f1, best_state = -1.0, None

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss = train_one_epoch(epoch)
    val_metrics, _, _ = evaluate(val_loader)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(val_metrics["loss"])
    history["val_acc"].append(val_metrics["acc"])
    history["val_f1"].append(val_metrics["f1"])

    print(f"Epoch {epoch}/{EPOCHS} ({time.time() - t0:.0f}s) | "
          f"train loss {tr_loss:.4f} | val loss {val_metrics['loss']:.4f} | "
          f"val acc {val_metrics['acc']:.4f} | val F1 {val_metrics['f1']:.4f}")

    if val_metrics["f1"] > best_f1:
        best_f1 = val_metrics["f1"]
        best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
        print("  -> new best model")

model.load_state_dict(best_state)
model.to(DEVICE)
print(f"\nBest validation F1: {best_f1:.4f}")

In [ ]:
epochs_range = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, history["train_loss"], marker="o", label="Train loss")
axes[0].plot(epochs_range, history["val_loss"], marker="o", label="Validation loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].set_title("Loss"); axes[0].legend()

axes[1].plot(epochs_range, history["val_acc"], marker="o", label="Val accuracy")
axes[1].plot(epochs_range, history["val_f1"], marker="o", label="Val F1")
axes[1].set_xlabel("Epoch"); axes[1].set_title("Validation metrics"); axes[1].legend()

for ax in axes:
    ax.set_xticks(list(epochs_range))
plt.tight_layout()
plt.show()

## 9. Evaluation on the Held-Out Test Set

The test set was never used for training or checkpoint selection, so it gives an unbiased estimate of generalisation. We report:

- **Accuracy**: overall fraction correct (meaningful here since classes are balanced)
- **Precision / Recall / F1** for the sarcastic class
- A **confusion matrix** to see which kinds of mistakes the model makes

In [ ]:
test_metrics, test_preds, test_labels = evaluate(test_loader)
print({k: round(v, 4) for k, v in test_metrics.items()})
print()
print(classification_report(test_labels, test_preds, target_names=["Not sarcastic", "Sarcastic"], digits=4))

## Context Ablation: Parent Comment

The main run follows `USE_PARENT`. The next cell evaluates the other setting on the same train/validation/test rows. It retrains with matching hyperparameters and chooses a checkpoint by validation F1; test metrics are calculated once for each setting.


In [ ]:
def run_context_experiment(use_parent):
    global USE_PARENT, train_ds, val_ds, test_ds, train_loader, val_loader, test_loader
    global model, optimizer, scheduler, scaler
    USE_PARENT = use_parent
    set_seed(SEED)
    train_ds, val_ds, test_ds = [SarcasmDataset(d) for d in (train_df, val_df, test_df)]
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collator)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collator)
    model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    groups = [
        {"params": [p for n,p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": WEIGHT_DECAY},
        {"params": [p for n,p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
    ]
    optimizer = AdamW(groups, lr=LEARNING_RATE)
    steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * steps), steps)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    best_f1, best_state = -1.0, None
    for epoch in range(1, EPOCHS + 1):
        train_one_epoch(epoch)
        val_metrics, _, _ = evaluate(val_loader)
        if val_metrics["f1"] > best_f1:
            best_f1 = val_metrics["f1"]
            best_state = copy.deepcopy({k: v.cpu() for k,v in model.state_dict().items()})
    model.load_state_dict(best_state); model.to(DEVICE)
    metrics, preds, labels = evaluate(test_loader)
    out_dir = f"bert_sarcasm_{'with' if use_parent else 'without'}_parent"
    model.save_pretrained(out_dir); tokenizer.save_pretrained(out_dir)
    return metrics, preds, labels

current_parent_setting = USE_PARENT
current_context_metrics, current_context_preds = test_metrics, test_preds
other_context_metrics, other_context_preds, _ = run_context_experiment(not current_parent_setting)
context_comparison = pd.DataFrame([
    {"use_parent": current_parent_setting, "accuracy": current_context_metrics["acc"], "f1": current_context_metrics["f1"]},
    {"use_parent": not current_parent_setting, "accuracy": other_context_metrics["acc"], "f1": other_context_metrics["f1"]},
])
context_comparison

### Baseline vs. BERT

Both models use the same held-out examples. BERT latency includes tokenization and batched forward passes; report hardware and batch size with latency.

In [ ]:
@torch.no_grad()
def timed_bert_test(frame):
    ds = SarcasmDataset(frame)
    loader = DataLoader(ds, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collator)
    model.eval(); preds = []
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    start = time.perf_counter()
    for batch in loader:
        batch = move(batch); batch.pop("labels")
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(**batch).logits
        preds.extend(logits.argmax(dim=-1).cpu().tolist())
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    return np.asarray(preds), (time.perf_counter()-start)*1000/len(frame)

bert_preds_timed, bert_latency_ms = timed_bert_test(test_df)
_, _, bert_f1, _ = precision_recall_fscore_support(test_df["label"], bert_preds_timed, average="binary", zero_division=0)
comparison = pd.DataFrame([
    {"model": "TF-IDF + Logistic Regression", "accuracy": accuracy_score(test_df["label"], baseline_preds), "f1": baseline_f1, "inference_ms_per_comment": baseline_latency_ms},
    {"model": f"BERT (parent={USE_PARENT})", "accuracy": accuracy_score(test_df["label"], bert_preds_timed), "f1": bert_f1, "inference_ms_per_comment": bert_latency_ms},
])
comparison

In [ ]:
cm = confusion_matrix(test_labels, test_preds)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
labels = ["Not sarcastic", "Sarcastic"]
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title("Confusion matrix (counts)")
sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=axes[1])
axes[1].set_title("Confusion matrix (row-normalised)")
for ax in axes:
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.show()

## 10. Error Analysis

Numbers only tell part of the story. Looking at the **most confident wrong predictions** helps reveal what the model finds hard, for example very short comments, sarcasm that depends on the parent comment, or non-sarcastic comments that merely sound exaggerated.

In [ ]:
@torch.no_grad()
def predict_proba(frame):
    ds = SarcasmDataset(frame.assign(label=0))
    loader = DataLoader(ds, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collator)
    model.eval()
    probs = []
    for batch in loader:
        batch = move(batch); batch.pop("labels")
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(**batch).logits
        probs.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu())
    return torch.cat(probs).numpy()

test_eval = test_df.copy()
test_eval["p_sarcastic"] = predict_proba(test_df)
test_eval["pred"] = (test_eval["p_sarcastic"] > 0.5).astype(int)
test_eval["confidence"] = np.where(test_eval["pred"] == 1, test_eval["p_sarcastic"], 1 - test_eval["p_sarcastic"])

wrong = test_eval[test_eval["pred"] != test_eval["label"]].sort_values("confidence", ascending=False)
print(f"Misclassified: {len(wrong)} / {len(test_eval)}\n")

for _, r in wrong.head(10).iterrows():
    true = "sarcastic" if r["label"] == 1 else "not sarcastic"
    print(f"[true: {true} | confidence in wrong answer: {r['confidence']:.2f}] r/{r['subreddit']}")
    print("   ", r["comment"][:250])
    print()

In [ ]:
# Does accuracy depend on comment length?
test_eval["n_words"] = test_eval["comment"].str.split().str.len()
test_eval["correct"] = (test_eval["pred"] == test_eval["label"]).astype(int)
test_eval["len_bucket"] = pd.cut(test_eval["n_words"], bins=[0, 5, 10, 20, 40, 1000],
                                 labels=["1-5", "6-10", "11-20", "21-40", "40+"])
by_len = test_eval.groupby("len_bucket", observed=True)["correct"].agg(["mean", "count"])
by_len["mean"].plot.bar(figsize=(6, 3.5), color="tab:green")
plt.ylabel("Accuracy"); plt.xlabel("Comment length (words)"); plt.ylim(0.5, 1.0)
plt.title("Test accuracy by comment length"); plt.xticks(rotation=0); plt.show()
by_len

### Error Slices by Reddit Metadata

These held-out slices are descriptive diagnostics. Small groups are filtered, and score quantiles are computed within the test set.

In [ ]:
test_eval["error"] = (test_eval["pred"] != test_eval["label"]).astype(int)
MIN_SLICE_N = 50
if "subreddit" in test_eval:
    subreddit_errors = (test_eval.groupby("subreddit", dropna=False)
                        .agg(n=("error", "size"), error_rate=("error", "mean"), accuracy=("correct", "mean"))
                        .query("n >= @MIN_SLICE_N").sort_values("error_rate", ascending=False))
    display(subreddit_errors.head(15))
for col in ["score", "ups", "downs"]:
    if col in test_eval:
        test_eval[f"{col}_bucket"] = pd.qcut(pd.to_numeric(test_eval[col], errors="coerce"), q=5, duplicates="drop")
        summary = (test_eval.groupby(f"{col}_bucket", observed=True)
                   .agg(n=("error", "size"), error_rate=("error", "mean"), accuracy=("correct", "mean")))
        print(f"\nError rate by {col} quantile")
        display(summary)
if "subreddit" in test_eval:
    plot_data = subreddit_errors.head(12).sort_values("error_rate")
    if len(plot_data):
        plot_data["error_rate"].plot.barh(figsize=(8, 5), color="tab:orange")
        plt.xlabel("Test error rate"); plt.title("Subreddits with highest error rate (min n=50)")
        plt.tight_layout(); plt.show()

## 11. Inference on Custom Text and Saving the Model

A small helper to score any new comment, followed by saving the fine-tuned model and tokenizer so they can be reloaded later with `from_pretrained`.

In [ ]:
@torch.no_grad()
def predict_sarcasm(texts, parents=None):
    if isinstance(texts, str):
        texts = [texts]
    if USE_PARENT:
        parents = parents or [""] * len(texts)
        enc = tokenizer(parents, texts, truncation=True, max_length=MAX_LEN,
                        padding=True, return_tensors="pt")
    else:
        enc = tokenizer(texts, truncation=True, max_length=MAX_LEN,
                        padding=True, return_tensors="pt")
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    model.eval()
    probs = torch.softmax(model(**enc).logits.float(), dim=-1)[:, 1].cpu().numpy()
    return [{"text": t, "p_sarcastic": float(p), "label": "sarcastic" if p > 0.5 else "not sarcastic"}
            for t, p in zip(texts, probs)]

examples = [
    "Oh wow, what a surprise, the government wasted money again.",
    "Thanks for the detailed explanation, that really helped me understand.",
    "Yeah, because that has worked so well in the past.",
    "I just got back from a hike and the view was amazing.",
    "Great, another meeting that could have been an email.",
    "The recipe needs two cups of flour and one egg.",
]
for r in predict_sarcasm(examples):
    print(f"{r['p_sarcastic']:.2f}  {r['label']:<14} | {r['text']}")

In [ ]:
SAVE_DIR = "bert_sarcasm_model"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved to", os.path.abspath(SAVE_DIR))

# Reload later with:
# model = BertForSequenceClassification.from_pretrained(SAVE_DIR)
# tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)

## Summary

This notebook compares TF-IDF + Logistic Regression with fine-tuned BERT on a shared stratified split, tests parent-comment context, and audits held-out errors by subreddit and voting metadata as well as comment length. Interpret subgroup differences cautiously: the dataset is historically sampled and labeled through Reddit conventions, and slice rates can reflect topic and sampling mix rather than model bias alone.

Record `N_SAMPLES`, seed, epochs, device, and context setting alongside reported metrics. Latency depends on hardware and batch size. Report whether context helped based on measured test results rather than assuming it will.
